# **Project 3: Visualization of Biological Data**

In this third project we will investigate the fundamentals of data visualization in Python from a practical perspective, applying them to the generation of charts, maps, and network visualizations from biological data commonly used in bioinformatics and computational biology.

The project is structured around four biological datasets, each explored using a different visualization approach:

- **Task 1 (matplotlib):** Static Plots: Differential gene expression in cancer (TCGA)
- **Task 2 (matplotlib interactive):** Dynamic Plots: Genomic variant surveillance across populations
- **Task 3 (geopandas + contextily):** Maps: Global biodiversity and species occurrence data (GBIF)
- **Task 4 (networkx):** Graphs: Metabolic pathway networks (KEGG-inspired)

In the folder `data/` you will find all necessary files.

<style>
.info_note-box {
  background-color: #f1f369;
  color: #3e3f05e0;
  border: 1px solid #f1f369;
  padding: 10px 15px;
  border-radius: 5px;
  margin: 10px 0px;
}
</style>
<div class="info_note-box">
  <p><b>Note:</b></p>
  <ul>
    <li> ...
  </ul>
</div>

<style>
.info-box {
  background-color: #D9EDF7;
  color: #31708F;
  border: 1px solid #BCE8F1;
  padding: 10px 15px;
  border-radius: 5px;
  margin: 10px 0;
}
</style>
<div class="info-box">
  <p><b>Allowed libraries:</b> 
  <ul>
  <li>You may use the following libraries: <code>numpy</code>, <code>pandas</code>, <code>matplotlib</code>, <code>geopandas</code>, <code>contextily</code>, <code>networkx</code>, <code>scipy</code>.</li>
   <li>For Task 3 only: <code>shapely</code> is also allowed for geometric operations.</li>
  </ul>
</div>

<style>
.submit-box {
  background-color: #f7d9db;
  color: #8f3131;
  border: 1px solid #f1bcbf;
  padding: 10px 15px;
  border-radius: 5px;
  margin: 10px 0;
}
</style>
<div class="submit-box">
  <p><b>Submission Instructions:</b> 
  <ul>
  <li>All solutions must be implemented in the provided file <code>G[XX]_project3.py</code>.
    <ul>
    <li>where <code>[XX]</code> must be replaced by the group number
    <li>For example: the group <i><b>G_01</b></i> must submit a file named: <code>G01_project3.py</code>
    </ul>
  <li>The project must be completed in <b>groups of two/three students</b>.
  <li>The project must be submitted on Moodle later than <i><b>May 31</b></i>.
  </ul>
</div>


## **Task 1 (T1):** Static Plots — Differential Gene Expression (TCGA)

Gene expression analysis is at the heart of cancer genomics. In Project 2 you computed differential expression numerically — here we visualise it. The same biological question motivates both: which genes behave differently in tumour tissue compared to normal tissue, and how can we communicate that effectively?

You are provided with two files:

- `data/expression_matrix.csv`: an RNA-seq expression matrix in $log_2$ TPM units. Rows are genes (Ensembl IDs), columns are samples. The first column is `gene_id`.
- `data/sample_metadata.csv`: maps each sample ID to its condition (`tumor` or `normal`) and cancer type (e.g., `BRCA`, `LUAD`, `COAD`).

The following code reads both files and computes the values you will need:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# Read expression matrix and samples
expr = pd.read_csv('data/expression_matrix.csv', index_col='gene_id')
metadata = pd.read_csv('data/sample_metadata.csv', index_col='sample_id')

# Align samples
samples = expr.columns.intersection(metadata.index)
expr = expr[samples]
metadata = metadata.loc[samples]

tumor_samples  = metadata[metadata['condition'] == 'tumor'].index
normal_samples = metadata[metadata['condition'] == 'normal'].index

# Per-gene statistics
mean_tumor  = expr[tumor_samples].mean(axis=1)
mean_normal = expr[normal_samples].mean(axis=1)
log2fc      = mean_tumor - mean_normal   # log2 fold-change (already in log2 space)

# Per-gene p-value (Welch t-test, one per gene)
pvalues = expr.apply(
    lambda row: stats.ttest_ind(
        row[tumor_samples], row[normal_samples], equal_var=False
    ).pvalue,
    axis=1
)
neg_log10_p = -np.log10(pvalues.clip(lower=1e-300))

print(f"Genes: {expr.shape[0]}  |  Tumor samples: {len(tumor_samples)}  |  Normal samples: {len(normal_samples)}")
```

### **T1.1** - Volcano Plot

A **volcano plot** is one of the most iconic visualisations in genomics. It plots statistical significance ($−log_{10}$ $p$-value) on the $Y$-axis against effect size ($log_2$ fold-change) on the $X$-axis, allowing rapid identification of genes that are both statistically significant and biologically meaningful.

Your plot must include:

- Each gene as a point, coloured by significance category:
  - **Red**: significantly overexpressed in tumour ($log_2(FC) > 1$ **and** $−log_{10}(p) > 2)
  - **Blue**: significantly underexpressed in tumour ($log_2(FC) < −1$ **and** $−log_{10}(p) > 2$)
  - **Grey**: not significant
- Dashed vertical lines at $log_2(FC) = \pm 1$ and a dashed horizontal line at $−log_{10}(p) = 2$ (corresponding to $p = 0.01$).
- Axis labels, a title, and a legend.

For full marks, the plot should also:
- Annotate the *top 5* most significant genes (by $−log_{10}(p)$) with their gene ID.
- Display a small summary text (e.g., "47 up / 31 down") in a corner of the plot.

In [ ]:
def volcanoPlot(ax, log2fc, neg_log10_p):
    """
    Draws a volcano plot on the given Axes.
    Colours: red = overexpressed, blue = underexpressed, grey = not significant.
    """
    # TODO


# Test
fig, ax = plt.subplots(figsize=(8, 6))
volcanoPlot(ax, log2fc, neg_log10_p)
plt.tight_layout()
plt.savefig('volcano_plot.png', dpi=150)
plt.show()

### **T1.2** — Expression Heatmap

A **heatmap** of expression values across samples is a standard way to reveal sample clustering and gene co-regulation patterns.

Complete the function `expressionHeatmap`, which draws a heatmap of the **top 20 most differentially expressed genes** (by absolute $log_2(FC)$) across all samples. The function receives the expression matrix `expr`, the metadata `metadata`, and draws the heatmap on the provided `Axes`.

Your heatmap must:

- Show genes as rows and samples as columns.
- **Order samples** so that tumour samples appear before normal samples (within each group, order by cancer type).
- **Order genes** by $log_2(FC)$ (most overexpressed at top, most underexpressed at bottom).
- Use a diverging colourmap (e.g., `RdBu_r`) centred at zero after z-score normalisation per gene.
- Include a colour bar, gene ID labels on the $Y$-axis, and a header annotation distinguishing *tumour* from *normal* columns.

In [ ]:
def expressionHeatmap(ax, expr, metadata):
    """
    Draws a heatmap of the top 20 most differentially expressed genes.
    Samples ordered: tumour (by cancer type) then normal. Genes ordered by log2FC.
    Z-score normalised per gene, diverging colourmap.
    """
    # TODO


# Test
fig, ax = plt.subplots(figsize=(12, 7))
expressionHeatmap(ax, expr, metadata)
plt.tight_layout()
plt.savefig('expression_heatmap.png', dpi=150)
plt.show()

### **T1.3** — Combined Figure

Complete the function `expressionSummaryFigure`, which produces a single publication-quality figure combining both visualisations above side by side (volcano plot on the left, heatmap on the right) with a shared title and appropriate overall formatting.

In [ ]:
def expressionSummaryFigure(expr, metadata):
    """
    Produces a combined figure: volcano plot (left) + heatmap (right).
    Saves to 'expression_summary.png'.
    """
    # TODO

# Test
expressionSummaryFigure(expr, metadata)

## **Task 2 (T2):** Dynamic Plots — Genomic Variant Surveillance

Tracking the emergence and spread of genomic variants across human populations is central to epidemiology and public health genomics. Here we work with a dataset of **single nucleotide polymorphism (SNP) frequencies** across populations and chromosomes over time, inspired by the 1000 Genomes Project and gnomAD.

You are provided with the file `data/snp_surveillance.csv`, which contains the following columns:

| Column | Description |
|--------|-------------|
| `snp_id` | SNP identifier (e.g., `rs12345`) |
| `chromosome` | Chromosome (1–22, X) |
| `population` | Population code (e.g., `EUR`, `AFR`, `EAS`, `SAS`, `AMR`) |
| `year` | Year of the survey (2015–2023) |
| `allele_frequency` | Frequency of the alternative allele (0–1) |
| `consequence` | Functional consequence (e.g., `missense`, `synonymous`, `intronic`, `stop_gained`) |
| `gene` | Gene where the SNP falls |

The following code reads the file:

```python
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import RadioButtons, CheckButtons

snps = pd.read_csv('data/snp_surveillance.csv')
populations = sorted(snps['population'].unique())
chromosomes = sorted(snps['chromosome'].unique(), key=lambda x: (int(x) if x.isdigit() else 100, x))
consequences = sorted(snps['consequence'].unique())
years = sorted(snps['year'].unique())

print(snps.head())
print(f"\n{len(snps)} records | {snps['snp_id'].nunique()} unique SNPs | {len(populations)} populations")
```

### **T2.1** — Per-Population Allele Frequency Trend

Complete the function `drawPopulationTrend`, which receives an `Axes` object, a chromosome string, and a consequence type, and draws a **line plot** showing the **mean allele frequency per year** for each population, restricted to SNPs on that chromosome with that consequence type.

- Each population should be drawn as a separate line with a distinct colour and marker.
- Include axis labels, a title, and a legend.
- Handle the case where no data is available for the combination (display a message on the axes).

In [ ]:
def drawPopulationTrend(ax, chromosome, consequence):
    """
    Draws mean allele frequency over time (per population) for SNPs
    on the given chromosome with the given consequence type.
    """
    # TODO

# Test
fig, ax = plt.subplots(figsize=(9, 5))
drawPopulationTrend(ax, '17', 'missense')
plt.tight_layout()
plt.savefig('population_trend.png', dpi=150)
plt.show()

### **T2.2** — Interactive Surveillance Dashboard

Complete the function `drawSurveillanceDashboard`, which creates an **interactive matplotlib figure** for exploring the SNP surveillance data. The dashboard must include:

- A main plot area showing the population trend (as in **T2.1**).
- A `RadioButtons` widget to select the **chromosome** (chromosomes 1–22 and X).
- A `RadioButtons` widget to select the **consequence type** (`missense`, `synonymous`, `intronic`, `stop_gained`).

Selecting a chromosome or consequence type must **dynamically update** the main plot.

**For full marks**, the dashboard should also:
- Include a secondary panel showing the **allele frequency distribution** (histogram or violin plot) for the current selection, updated alongside the main plot.
- Display summary statistics (number of SNPs, mean frequency) as a text annotation.

In [ ]:
def drawSurveillanceDashboard():
    """
    Interactive matplotlib dashboard for SNP surveillance data.
    RadioButtons for chromosome and consequence type selection.
    Plot updates dynamically on selection.
    """
    # TODO

# Test
drawSurveillanceDashboard()

## **Task 3 (T3):** Maps — Global Species Occurrence (GBIF)

The [Global Biodiversity Information Facility (GBIF)](https://www.gbif.org/) is the world's largest open database of species occurrence records, aggregating hundreds of millions of observations contributed by institutions and citizen scientists worldwide. Mapping occurrence data is a fundamental task in conservation biology and biogeography.

You are provided with two files:

- `data/species_occurrences.csv`: a dataset of occurrence records for a set of endangered mammal species, downloaded from GBIF. The relevant columns are:

| Column | Description |
|--------|-------------|
| `species` | Scientific name (e.g., *Panthera leo*) |
| `decimalLatitude` | Latitude |
| `decimalLongitude` | Longitude |
| `year` | Year of observation |
| `country` | Country code (ISO 3166) |
| `basisOfRecord` | Record type (e.g., `HUMAN_OBSERVATION`, `PRESERVED_SPECIMEN`) |

- `data/countries.geojson`: *GeoJSON* file with country polygons (world map), with a `ISO_A2` field for country codes.

The following code reads both files:

```python
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
from shapely.geometry import Point

occurrences = pd.read_csv('data/species_occurrences.csv').dropna(subset=['decimalLatitude','decimalLongitude'])
world       = gpd.read_file('data/countries.geojson')  # ISO3166-1-Alpha-3 column

species_list = sorted(occurrences['species'].unique())
print(f"{len(occurrences)} occurrence records | {len(species_list)} species")
print(occurrences.head())
```


### **T3.1** — Species Richness Choropleth Map

**Species richness** (the number of distinct species recorded in a region) is a primary measure of biodiversity. A choropleth map colours each country proportionally to a variable of interest.

Complete the function `drawSpeciesRichness`, which draws a **world choropleth map** coloured by the number of distinct endangered species recorded in each country.

- Join the occurrence data with the country polygons using the `country` / `ISO_A2` fields.
- Countries with no occurrences should appear in a neutral colour (e.g., light grey).
- Include a colour bar, a title, and a note on the data source.
- Add a basemap using `contextily`.

In [ ]:
def drawSpeciesRichness(ax, occurrences, world):
    """
    Draws a choropleth map of species richness (n distinct species) per country.
    """
    # TODO

# Test
fig, ax = plt.subplots(figsize=(14, 7))
drawSpeciesRichness(ax, occurrences, world)
plt.tight_layout()
plt.savefig('species_richness.png', dpi=150)
plt.show()

### **T3.2** — Per-Species Occurrence Map

Complete the function `drawSpeciesOccurrences`, which draws a **point map** for a given species, showing all occurrence records as points on a map.

- Convert the occurrence table to a `GeoDataFrame` using the latitude/longitude columns.
- Each point should be coloured by `basisOfRecord` (e.g., field observation vs. museum specimen) and sized proportionally to recency (more recent observations larger).
- Restrict the map extent to the bounding box of the occurrences (with a small margin).
- Add a basemap using `contextily`.
- Include a legend and a title.

In [ ]:
def drawSpeciesOccurrences(ax, occurrences, species_name):
    """
    Draws a point map of all occurrence records for the given species.
    Point colour encodes basisOfRecord; point size encodes recency.
    """
    # TODO

fig, ax = plt.subplots(figsize=(10, 8))
drawSpeciesOccurrences(ax, occurrences, 'Panthera leo')
plt.tight_layout()
plt.savefig('species_occurrences_map.png', dpi=150)
plt.show()

### **T3.3** — Temporal Spread Animation 

Complete the function `drawTemporalSpread`, which produces a **series of maps** (one per decade) showing how the recorded occurrences of a species have spread (or contracted) geographically over time. Save the maps as a multi-panel figure.

In [ ]:
def drawTemporalSpread(occurrences, species_name):
    """
    Draws a multi-panel map showing occurrence records per decade.
    Saves to 'temporal_spread.png'.
    """
    # TODO

# Test
 drawTemporalSpread(occurrences, 'Panthera leo')

## **Task 4 (T4):** Network Graphs — Metabolic Pathways

Cellular metabolism is organised into **pathways** — sequences and cycles of biochemical reactions connecting metabolites through enzyme-catalysed steps. These pathways are naturally represented as graphs, where nodes are metabolites (or reactions) and edges represent biochemical transformations. Visualising and analysing these networks is a core task in systems biology.

This task is inspired by the [KEGG Pathway Database](https://www.genome.jp/kegg/pathway.html), one of the primary repositories of metabolic pathway information. You are provided with a simplified graph of the central carbon metabolism (glycolysis, TCA cycle, and pentose phosphate pathway) for *Homo sapiens*.

You are provided with two files:

- `data/metabolites.csv`: nodes of the metabolic network.

| Column | Description |
|--------|-------------|
| `metabolite_id` | Unique identifier (e.g., `C00031`) |
| `name` | Common name (e.g., `Glucose`) |
| `pathway` | Pathway membership (`Glycolysis`, `TCA`, `PPP`, `Shared`) |
| `formula` | Chemical formula |

- `data/reactions.csv`: edges of the metabolic network.

| Column | Description |
|--------|-------------|
| `source` | Source metabolite ID |
| `target` | Target metabolite ID |
| `reaction_id` | Reaction identifier |
| `enzyme` | Enzyme name (EC number) |
| `reversible` | Whether the reaction is reversible (`True`/`False`) |
| `delta_G` | Standard Gibbs free energy change (kJ/mol) |

The following code reads both files and builds the graph:

```python
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

metabolites = pd.read_csv('data/metabolites.csv', index_col='metabolite_id')
reactions   = pd.read_csv('data/reactions.csv')

print(f"Metabolites: {len(metabolites)}")
print(f"Reactions: {len(reactions)}")
print(metabolites.head())
print(reactions.head())
```

### **T4.1** — Build the Metabolic Graph

Complete the function `buildMetabolicGraph`, which builds and returns a directed graph (`nx.DiGraph`) from the metabolites and reactions tables.

- Each metabolite is a node, with attributes `name`, `pathway`, and `formula`.
- Each reaction is a directed edge from `source` to `target`, with attributes `reaction_id`, `enzyme`, `reversible`, and `delta_G`.
- If a reaction is **reversible**, add edges in **both directions**.

In [ ]:
def buildMetabolicGraph(metabolites, reactions):
    """
    Returns a nx.DiGraph of the metabolic network.
    Nodes: metabolites with attributes name, pathway, formula.
    Edges: reactions with attributes reaction_id, enzyme, reversible, delta_G.
    Reversible reactions generate edges in both directions.
    """
    # TODO

# Test
G = buildMetabolicGraph(metabolites, reactions)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

### **T4.2** — Visualise the Metabolic Network

Complete the function `drawMetabolicNetwork`, which draws the metabolic network constructed in **T4.1** using `networkx`.

Your visualisation must respect the following requirements:

- **Node colour** encodes pathway membership (`Glycolysis`, `TCA`, `PPP`, `Shared`), using a consistent colour palette.
- **Node size** is proportional to the node's **degree** (more connected metabolites appear larger).
- **Edge colour** encodes the sign of ΔG: reactions that are thermodynamically favourable (ΔG < 0) in one colour, unfavourable (ΔG > 0) in another.
- **Edge style** distinguishes reversible (dashed) from irreversible (solid) reactions.
- Node labels show the metabolite short name (not the ID).
- Include a legend for node colours (pathway) and edge colours (ΔG sign).

For the layout, use `nx.kamada_kawai_layout` or `nx.spring_layout` — note that a purely random layout will obscure the biological structure. You may also manually adjust node positions to place the three pathways in visually distinct regions.

In [ ]:
def drawMetabolicNetwork(ax, G, metabolites):
    """
    Draws the metabolic network with pathway-coloured nodes,
    degree-scaled node sizes, and ΔG-coloured edges.
    """
    # TODO

# Test
fig, ax = plt.subplots(figsize=(14, 10))
drawMetabolicNetwork(ax, G, metabolites)
plt.tight_layout()
plt.savefig('metabolic_network.png', dpi=150)
plt.show()

### **T4.3** — Network Analysis

Complete the following three functions to characterise the metabolic network:

**T4.3a** — `centralMetabolites`: Returns a list of the **top-5 metabolites by betweenness centrality** (as tuples `(name, centrality)`), sorted by descending centrality. Betweenness centrality measures how often a node lies on shortest paths between other nodes — high-betweenness metabolites are critical hubs in the network.

In [ ]:
def centralMetabolites(G, metabolites):
    """
    Returns [(name, centrality)] for the top-5 metabolites by betweenness centrality.
    """
    # TODO

# Test
print("Top-5 central metabolites:")
for name, bc in centralMetabolites(G, metabolites):
    print(f"  {name}: {bc:.4f}")

**T4.3b** — `findShortestPathway`: Given two metabolite IDs, returns the **shortest directed path** between them as a list of metabolite names (using common names), or `None` if no path exists. This corresponds to finding the minimal sequence of reactions connecting two metabolites.

In [ ]:
def findShortestPathway(G, metabolites, source_id, target_id):
    """
    Returns a list of metabolite names along the shortest directed path
    from source_id to target_id, or None if no path exists.
    """
    # TODO

# Test
path = findShortestPathway(G, metabolites, 'C00031', 'C00022')  # Glucose → Pyruvate
print(f"Glucose → Pyruvate: {' → '.join(path) if path else 'No path'}")

**T4.3c** — `pathwayConnectivity`: Returns a dictionary `{pathway: n_cross_edges}` counting how many directed edges **cross between pathways** (i.e., source and target belong to different pathways, excluding `Shared` nodes). This measures the degree of integration between the three metabolic modules.

In [ ]:
def pathwayConnectivity(G, metabolites):
    """
    Returns {pathway_pair: n_edges} for edges crossing between pathways.
    Pathway pair is a frozenset of two pathway names (excluding 'Shared').
    """
    # TODO

# Test
print("Cross-pathway edges:")
for pair, n in pathwayConnectivity(G, metabolites).items():
    print(f"  {' ↔ '.join(pair)}: {n} edge(s)")